# Normalize Spectral Features

Apply z-score normalization to the 7 spectral features in a parquet feature file.
Pitch columns (`f0_hz`, `f0_confidence`) are left untouched.

Normalization stats (mean/std) are fitted on a **reference** parquet and then applied
to any number of target parquet files so that every dataset lives in the same
feature space for downstream comparison.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent.parent.parent
print(f"Project root: {PROJECT_ROOT}")

Project root: /m/home/home3/37/thieun1/unix/Project/final_project


## Configuration

Edit this cell to point at the parquet files you want to normalize.

In [7]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Fit mean/std on this file
REFERENCE_PARQUET = DATA_DIR / "bach_features.parquet"

# Apply those stats to every file in this list (reference included if desired)
TARGET_PARQUETS = [
    # DATA_DIR / "trainset_features.parquet",
    DATA_DIR / "baseline_features.parquet",
    # DATA_DIR / "sms_features.parquet",
    DATA_DIR / "bach_features.parquet",
    DATA_DIR / "transfered_features.parquet",
    DATA_DIR / "transfered_noreverb_features.parquet",
]

# Output directory for normalized parquets ('<name>_norm.parquet')
OUT_DIR = DATA_DIR / "normalized"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# The 7 spectral features to normalize — pitch columns are excluded on purpose
FEATURES_TO_NORMALIZE = [
    "spectral_centroid",
    "spectral_crest",
    "spectral_decrease",
    "spectral_flatness",
    "spectral_roll_off",
    "spectral_skewness",
    "spectral_spread",
]
PITCH_COLS = ["f0_hz", "f0_confidence"]  # kept as-is

print(f"Reference: {REFERENCE_PARQUET.name}")
print(f"Targets  : {[p.name for p in TARGET_PARQUETS]}")
print(f"Features : {FEATURES_TO_NORMALIZE}")

Reference: bach_features.parquet
Targets  : ['baseline_features.parquet', 'bach_features.parquet', 'transfered_features.parquet', 'transfered_noreverb_features.parquet']
Features : ['spectral_centroid', 'spectral_crest', 'spectral_decrease', 'spectral_flatness', 'spectral_roll_off', 'spectral_skewness', 'spectral_spread']


## 1. Fit normalization stats on the reference file

We use z-score normalization: `(x - mean) / std`.
Stats are computed over all frames, ignoring NaNs.

In [8]:
ref_df = pd.read_parquet(REFERENCE_PARQUET)
print(f"Reference shape: {ref_df.shape}")

missing = [c for c in FEATURES_TO_NORMALIZE if c not in ref_df.columns]
if missing:
    raise KeyError(f"Reference parquet is missing columns: {missing}")

stats = pd.DataFrame({
    "mean": ref_df[FEATURES_TO_NORMALIZE].mean(skipna=True),
    "std":  ref_df[FEATURES_TO_NORMALIZE].std(skipna=True, ddof=0),
})

# Guard against zero-variance features
zero_std = stats.index[stats["std"] == 0].tolist()
if zero_std:
    print(f"WARNING: zero-variance features, leaving them unscaled: {zero_std}")
    stats.loc[zero_std, "std"] = 1.0

stats.round(4)

Reference shape: (50347, 10)


,mean,std
spectral_centroid,1721.8436,407.3948
spectral_crest,78.3068,42.4556
spectral_decrease,0.0039,0.2076
spectral_flatness,0.1554,0.0610
spectral_roll_off,4522.5521,820.4087
spectral_skewness,1.6151,4.7961
spectral_spread,1391.6614,183.7231


In [9]:
stats_path = OUT_DIR / "normalization_stats.csv"
stats.to_csv(stats_path)
print(f"Saved stats: {stats_path}")

Saved stats: /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/normalized/normalization_stats.csv


## 2. Apply stats to every target parquet

Each target gets the same mean/std applied to the 7 spectral features.
Pitch and any metadata columns are copied through unchanged.

In [10]:
def normalize_parquet(src: Path, stats: pd.DataFrame, out_dir: Path) -> Path:
    df = pd.read_parquet(src)

    missing = [c for c in stats.index if c not in df.columns]
    if missing:
        raise KeyError(f"{src.name} missing feature columns: {missing}")

    df_out = df.copy()
    df_out[stats.index] = (
        (df[stats.index] - stats["mean"]) / stats["std"]
    ).astype(np.float64)

    out_path = out_dir / f"{src.stem}_norm.parquet"
    df_out.to_parquet(out_path, index=False)
    return out_path


written = []
for src in TARGET_PARQUETS:
    if not src.exists():
        print(f"  SKIP (missing): {src}")
        continue
    out_path = normalize_parquet(src, stats, OUT_DIR)
    print(f"  {src.name} -> {out_path.name}  ({out_path.stat().st_size / 1e6:.1f} MB)")
    written.append(out_path)

print(f"\nWrote {len(written)} normalized parquet file(s) to {OUT_DIR}")

  baseline_features.parquet -> baseline_features_norm.parquet  (8.5 MB)
  bach_features.parquet -> bach_features_norm.parquet  (4.0 MB)
  transfered_features.parquet -> transfered_features_norm.parquet  (8.5 MB)
  transfered_noreverb_features.parquet -> transfered_noreverb_features_norm.parquet  (8.5 MB)

Wrote 4 normalized parquet file(s) to /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/normalized


## 3. Sanity check

After applying the stats fit on the reference file, the reference itself should
have approximately zero mean and unit std on the normalized features.
Other files will differ — that is the whole point.

In [11]:
for out_path in written:
    df = pd.read_parquet(out_path)
    summary = df[FEATURES_TO_NORMALIZE].agg(["mean", "std"]).T.round(3)
    print(f"\n{out_path.name}")
    print(summary)
    for col in PITCH_COLS:
        if col in df.columns:
            print(f"  {col}: min={df[col].min():.2f}  max={df[col].max():.2f}  (untouched)")


baseline_features_norm.parquet
                    mean    std
spectral_centroid  1.210  1.004
spectral_crest    -1.194  0.438
spectral_decrease  0.009  0.018
spectral_flatness  3.355  1.510
spectral_roll_off  1.023  0.872
spectral_skewness -0.117  0.074
spectral_spread    1.139  0.919
  f0_hz: min=31.89  max=1999.83  (untouched)
  f0_confidence: min=0.00  max=0.99  (untouched)

bach_features_norm.parquet
                   mean  std
spectral_centroid   0.0  1.0
spectral_crest     -0.0  1.0
spectral_decrease  -0.0  1.0
spectral_flatness  -0.0  1.0
spectral_roll_off   0.0  1.0
spectral_skewness  -0.0  1.0
spectral_spread     0.0  1.0
  f0_hz: min=0.00  max=1993.94  (untouched)
  f0_confidence: min=0.02  max=0.95  (untouched)

transfered_features_norm.parquet
                    mean    std
spectral_centroid  1.312  1.186
spectral_crest    -1.123  0.434
spectral_decrease -0.007  0.062
spectral_flatness  4.613  2.371
spectral_roll_off  1.700  1.238
spectral_skewness -0.088  0.084
spectra